# Are the gaps bigger than chance? (Phase 4: statistics)

Phase 3 found gaps between groups. Here each one gets two checks: **is it bigger than chance
would explain**, and **how big is it** (with a 95% confidence interval). These are still
associations in past orders, not causes. Numbers come from the **analytics** tables built by
`python -m src.pipeline`.

**Contents:** 1. Setup · 2. How sure are we of one rate? · 3. Late vs on time ·
4. Speed on on-time orders · 5. Seller vs carrier, and routes · 6. Categories and the minimum group size

## 1. Setup
Same connection helper as the first notebook. **SciPy** is Python's standard library for
statistics; we use its tests and intervals instead of writing the formulas ourselves.

In [1]:
import math
import sys
from pathlib import Path

import pandas as pd
from scipy import stats
from sqlalchemy import text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # so "from src..." works when run from notebooks/
from src.db import get_engine

engine = get_engine()
pd.set_option("display.float_format", lambda x: f"{x:.4g}")   # 4 significant digits, so tiny p-values stay readable

def query(sql: str) -> pd.DataFrame:
    """Run one SQL query and return the result as a table."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 2. How sure are we of one rate?
The main KPI is the unhappy rate: 1-2 star orders ÷ reviewed orders. Our orders are one sample of
what Olist's customers do, so the rate would wobble a little with a different sample. A **95%
confidence interval** is the range the true rate probably lies in: if we repeated the sampling
many times, 95% of such ranges would contain it.

We use the **Wilson** interval (SciPy's `binomtest(...).proportion_ci`), which stays accurate for
small groups and rates near 0% or 100%. The simple textbook formula, rate ± 1.96 × standard error,
is shown next to it as a check.

In [2]:
def rate_ci(unhappy: int, reviewed: int) -> dict:
    """Unhappy rate with a 95% Wilson interval, plus the simple formula as a check (all in %)."""
    ci = stats.binomtest(unhappy, reviewed).proportion_ci(confidence_level=0.95, method="wilson")
    p = unhappy / reviewed
    se = math.sqrt(p * (1 - p) / reviewed)          # standard error: typical wobble of the rate
    return {"reviewed": reviewed, "unhappy": unhappy, "rate_pct": round(100 * p, 2),
            "wilson_low": round(100 * ci.low, 2), "wilson_high": round(100 * ci.high, 2),
            "simple_low": round(100 * (p - 1.96 * se), 2), "simple_high": round(100 * (p + 1.96 * se), 2)}

kpi = query("""
    SELECT count(*) FILTER (WHERE has_review) AS reviewed, count(*) FILTER (WHERE is_unhappy) AS unhappy
    FROM analytics.fact_order
""").iloc[0]
pd.DataFrame([rate_ci(int(kpi["unhappy"]), int(kpi["reviewed"]))], index=["all reviewed orders"])

,reviewed,unhappy,rate_pct,wilson_low,wilson_high,simple_low,simple_high
all reviewed orders,98673,14533,14.73,14.51,14.95,14.51,14.95


**Group size matters.** The same calculation for a small group from P1: orders 8+ days late whose
customer rated *after* the parcel arrived.

In [3]:
small = query("""
    SELECT count(*) AS reviewed, count(*) FILTER (WHERE is_unhappy) AS unhappy
    FROM analytics.fact_order
    WHERE order_status = 'delivered' AND days_late >= 8 AND has_review
      AND review_answer_timestamp >= order_delivered_customer_date
""").iloc[0]
pd.DataFrame([rate_ci(int(kpi["unhappy"]), int(kpi["reviewed"])),
              rate_ci(int(small["unhappy"]), int(small["reviewed"]))],
             index=["all reviewed orders", "8+ days late, rated after arrival"])

,reviewed,unhappy,rate_pct,wilson_low,wilson_high,simple_low,simple_high
all reviewed orders,98673,14533,14.73,14.51,14.95,14.51,14.95
"8+ days late, rated after arrival",76,23,30.26,21.09,41.33,19.93,40.59


## 3. Late vs on time (P1)
Two groups, two unhappy rates. Two questions:
- **Bigger than chance?** The **chi-square test** asks: if lateness had nothing to do with
  unhappiness, how likely is a gap this big just by luck? That chance is the **p-value**. Below 0.05
  is the usual line for "unlikely to be luck alone".
- **How big?** The gap in percentage points with a 95% interval, and the **risk ratio** (how many
  times more often late orders are unhappy) with SciPy's interval.

In [4]:
def compare(name: str, a_unhappy: int, a_n: int, b_unhappy: int, b_n: int) -> dict:
    """Group A vs group B: gap in points (95% interval), risk ratio (95% interval), chi-square p-value."""
    pa, pb = a_unhappy / a_n, b_unhappy / b_n
    se = math.sqrt(pa * (1 - pa) / a_n + pb * (1 - pb) / b_n)       # wobble of the gap
    table = [[a_unhappy, a_n - a_unhappy], [b_unhappy, b_n - b_unhappy]]
    p_value = stats.chi2_contingency(table).pvalue
    rr = stats.contingency.relative_risk(a_unhappy, a_n, b_unhappy, b_n)
    rr_ci = rr.confidence_interval(confidence_level=0.95)
    return {"comparison": name, "n_a": a_n, "rate_a_pct": round(100 * pa, 2),
            "n_b": b_n, "rate_b_pct": round(100 * pb, 2),
            "gap_points": round(100 * (pa - pb), 2),
            "gap_low": round(100 * (pa - pb - 1.96 * se), 2), "gap_high": round(100 * (pa - pb + 1.96 * se), 2),
            "risk_ratio": round(rr.relative_risk, 2), "rr_low": round(rr_ci.low, 2), "rr_high": round(rr_ci.high, 2),
            "p_value": float(p_value)}                                   # exact, so Holm can use it

In [5]:
# Your Step 2 query, plus the same counts for customers who rated AFTER the parcel arrived
p1 = query("""
    SELECT CASE WHEN days_late > 0 THEN 'late' ELSE 'on time' END AS promise,
           count(*)                           AS reviewed,
           count(*) FILTER (WHERE is_unhappy) AS unhappy,
           count(*) FILTER (WHERE review_answer_timestamp >= order_delivered_customer_date) AS reviewed_after,
           count(*) FILTER (WHERE is_unhappy AND review_answer_timestamp >= order_delivered_customer_date)
               AS unhappy_after
    FROM analytics.fact_order
    WHERE order_status = 'delivered' AND days_late IS NOT NULL AND has_review
    GROUP BY promise
""").set_index("promise")
late, on_time = p1.loc["late"], p1.loc["on time"]

pd.DataFrame([
    compare("late vs on time (all reviewed)",
            int(late.unhappy), int(late.reviewed), int(on_time.unhappy), int(on_time.reviewed)),
    compare("late vs on time (rated after arrival)",
            int(late.unhappy_after), int(late.reviewed_after), int(on_time.unhappy_after), int(on_time.reviewed_after)),
]).set_index("comparison").astype(object).T   # object: counts stay whole numbers

comparison,late vs on time (all reviewed),late vs on time (rated after arrival)
n_a,6381,1907
rate_a_pct,62.47,19.45
n_b,89443,89242
rate_b_pct,9.31,9.28
gap_points,53.16,10.18
gap_low,51.96,8.39
gap_high,54.36,11.96
risk_ratio,6.71,2.1
rr_low,6.53,1.91
rr_high,6.9,2.3


The gap is far bigger than chance in both versions. Counting only customers who rated after the
parcel arrived, the gap shrinks a lot but stays clear. The groups still differ in other ways
(routes, products, sellers), so this is a link, not proof that lateness is the reason.

## 4. Speed on on-time orders (P1)
Only orders that arrived on time, so a broken promise is kept out. Does a slower delivery still go
with more unhappiness? Each band gets its own interval; one chi-square test checks whether the four
bands differ more than luck would explain.

**Like with like:** slow deliveries are often long-distance ones, so the same bands are repeated
separately for same-state and different-state routes (single-seller orders, where we know the
seller's state). If the trend shows up inside both, distance alone doesn't explain it.

In [6]:
BAND = """CASE WHEN f.delivery_days <= 7 THEN '1. 0-7 days' WHEN f.delivery_days <= 14 THEN '2. 8-14 days'
               WHEN f.delivery_days <= 21 THEN '3. 15-21 days' ELSE '4. 22+ days' END"""
ON_TIME = "f.order_status = 'delivered' AND f.days_late <= 0 AND f.has_review"

speed = query(f"""
    SELECT {BAND} AS delivery_time, count(*) AS reviewed, count(*) FILTER (WHERE f.is_unhappy) AS unhappy
    FROM analytics.fact_order f WHERE {ON_TIME}
    GROUP BY delivery_time ORDER BY delivery_time
""")
bands = pd.DataFrame([rate_ci(int(r.unhappy), int(r.reviewed)) for r in speed.itertuples()],
                     index=speed["delivery_time"]).drop(columns=["simple_low", "simple_high"])
p_bands = stats.chi2_contingency(list(zip(speed["unhappy"], speed["reviewed"] - speed["unhappy"]))).pvalue
print(f"chi-square across the 4 bands: p = {p_bands:.1e}")
bands

chi-square across the 4 bands: p = 5.1e-108


,reviewed,unhappy,rate_pct,wilson_low,wilson_high
delivery_time,,,,,
1. 0-7 days,30476,2304,7.56,7.27,7.86
2. 8-14 days,37441,3333,8.9,8.62,9.19
3. 15-21 days,15473,1717,11.1,10.61,11.6
4. 22+ days,6053,970,16.03,15.12,16.97


In [7]:
fast, slow = speed.iloc[0], speed.iloc[-1]
pd.DataFrame([compare("22+ days vs 0-7 days (on time)", int(slow.unhappy), int(slow.reviewed),
                      int(fast.unhappy), int(fast.reviewed))]).set_index("comparison").astype(object).T   # object: counts stay whole numbers

comparison,22+ days vs 0-7 days (on time)
n_a,6053
rate_a_pct,16.03
n_b,30476
rate_b_pct,7.56
gap_points,8.47
gap_low,7.49
gap_high,9.44
risk_ratio,2.12
rr_low,1.98
rr_high,2.27


In [8]:
by_route = query(f"""
    SELECT CASE WHEN c.state = s.state THEN 'same state' ELSE 'different state' END AS route,
           {BAND} AS delivery_time, count(*) AS reviewed, count(*) FILTER (WHERE f.is_unhappy) AS unhappy
    FROM analytics.fact_order f
    JOIN analytics.dim_customer c ON c.customer_id = f.customer_id
    JOIN analytics.dim_seller s   ON s.seller_id = f.seller_id      -- single-seller orders only
    WHERE {ON_TIME}
    GROUP BY route, delivery_time ORDER BY route DESC, delivery_time
""")
rows = [{"route": r.route, "delivery_time": r.delivery_time, **rate_ci(int(r.unhappy), int(r.reviewed))}
        for r in by_route.itertuples()]
pd.DataFrame(rows).drop(columns=["simple_low", "simple_high"]).set_index(["route", "delivery_time"])

reviewed  unhappy  rate_pct  wilson_low  \
route           delivery_time                                            
same state      1. 0-7 days       20260     1368      6.75        6.41   
                2. 8-14 days      10137      922       9.1        8.55   
                3. 15-21 days      1802      253     14.04       12.51   
                4. 22+ days         292       70     23.97       19.43   
different state 1. 0-7 days        9619      655      6.81        6.32   
                2. 8-14 days      26817     2194      8.18        7.86   
                3. 15-21 days     13545     1393     10.28        9.78   
                4. 22+ days        5722      878     15.34       14.43   

                               wilson_high  
route           delivery_time               
same state      1. 0-7 days           7.11  
                2. 8-14 days          9.67  
                3. 15-21 days        15.72  
                4. 22+ days          29.19  
different state 1. 0-7 days           7.33  
                2. 8-14 days          8.52  
                3. 15-21 days        10.81  
                4. 22+ days           16.3

Slower on-time deliveries go with more unhappy customers, well beyond chance, and the rise appears
within same-state and within different-state routes. So it isn't only about distance. It is still
an association: slow orders may differ in other ways too (products, sellers, regions).

## 5. Seller vs carrier, and routes (P2)
P2 orders: delivered, one shared hand-over deadline, a hand-over date, no odd or impossible stage
dates (the same filter as `sql/questions/02_p2_stages_routes.sql`).
- **Late deliveries:** what share had the seller hand over on time? (One rate, with its interval.)
- **On-time deliveries:** is a late hand-over still linked with more unhappiness?
- **Routes, like with like:** different-state vs same-state routes, compared **within the same
  distance band**, so distance is held roughly equal.

In [9]:
P2 = """f.order_status = 'delivered' AND f.days_late IS NOT NULL AND f.has_one_deadline
        AND f.order_delivered_carrier_date IS NOT NULL AND NOT f.flag_odd_deadline
        AND NOT f.flag_handover_before_purchase AND NOT f.flag_delivered_before_handover"""
stages = query(f"""
    SELECT f.is_late, f.handover_late, count(*) AS orders,
           count(*) FILTER (WHERE f.has_review) AS reviewed, count(*) FILTER (WHERE f.is_unhappy) AS unhappy
    FROM analytics.fact_order f WHERE {P2}
    GROUP BY f.is_late, f.handover_late
""").set_index(["is_late", "handover_late"])

late_seller_ok, late_all = stages.loc[(True, False), "orders"], stages.loc[True, "orders"].sum()
share = stats.binomtest(int(late_seller_ok), int(late_all)).proportion_ci(method="wilson")
print(f"late deliveries with the seller on time: {late_seller_ok:,} of {late_all:,} = "
      f"{100 * late_seller_ok / late_all:.2f}% (95% interval {100 * share.low:.2f}-{100 * share.high:.2f}%)")

a, b = stages.loc[(False, True)], stages.loc[(False, False)]
pd.DataFrame([compare("on-time delivery: seller late vs seller on time",
                      int(a.unhappy), int(a.reviewed), int(b.unhappy), int(b.reviewed))]).set_index("comparison").astype(object).T   # object: counts stay whole numbers

late deliveries with the seller on time: 5,071 of 6,531 = 77.65% (95% interval 76.62-78.64%)


comparison,on-time delivery: seller late vs seller on time
n_a,4063
rate_a_pct,14.57
n_b,84865
rate_b_pct,8.91
gap_points,5.66
gap_low,4.56
gap_high,6.77
risk_ratio,1.64
rr_low,1.51
rr_high,1.77


In [10]:
routes = query(f"""
    SELECT CASE WHEN f.distance_km < 100 THEN '1. under 100 km' WHEN f.distance_km < 500 THEN '2. 100-499 km'
                WHEN f.distance_km < 1000 THEN '3. 500-999 km' ELSE '4. 1,000+ km' END AS distance,
           c.state = s.state AS same_state,
           count(*) AS reviewed, count(*) FILTER (WHERE f.is_unhappy) AS unhappy
    FROM analytics.fact_order f
    JOIN analytics.dim_customer c ON c.customer_id = f.customer_id
    JOIN analytics.dim_seller s   ON s.seller_id = f.seller_id
    WHERE {P2} AND f.has_review AND f.distance_km IS NOT NULL
    GROUP BY distance, same_state
""").set_index(["distance", "same_state"])

rows = []
for band in sorted(routes.index.get_level_values(0).unique()):
    if (band, True) in routes.index and (band, False) in routes.index:
        d, s = routes.loc[(band, False)], routes.loc[(band, True)]
        rows.append(compare(f"{band}: different vs same state",
                            int(d.unhappy), int(d.reviewed), int(s.unhappy), int(s.reviewed)))
pd.DataFrame(rows).set_index("comparison")[["n_a", "rate_a_pct", "n_b", "rate_b_pct",
                                            "gap_points", "gap_low", "gap_high", "p_value"]]

,n_a,rate_a_pct,n_b,rate_b_pct,gap_points,gap_low,gap_high,p_value
comparison,,,,,,,,
1. under 100 km: different vs same state,139,10.07,17418,10.1,-0.03,-5.05,5,1
2. 100-499 km: different vs same state,20506,13.46,15661,10.04,3.42,2.75,4.08,4.398e-23
3. 500-999 km: different vs same state,24322,13.09,840,8.69,4.4,2.44,6.35,0.0002331


`n_a` / `rate_a_pct` = different-state routes, `n_b` / `rate_b_pct` = same-state routes, in the same
distance band. Where the interval for the gap includes 0, the two route types can't be told apart
at that distance.

## 6. Categories and the minimum group size (P3)
**Minimum group size.** A rate is only worth comparing if its interval is reasonably narrow. At the
usual on-time unhappy rate, how many orders does a group need for its 95% interval to be within
about ±2.5 points? That number becomes the rule for every group in this project.

In [11]:
cats = query("""
    WITH single AS (
      SELECT order_id, min(product_id) AS product_id FROM analytics.fact_order_item
      GROUP BY order_id HAVING count(DISTINCT product_id) = 1
    )
    SELECT p.product_category AS category, count(*) AS reviewed_on_time,
           count(*) FILTER (WHERE f.is_unhappy) AS unhappy_on_time
    FROM single x
    JOIN analytics.fact_order f  ON f.order_id = x.order_id
    JOIN analytics.dim_product p ON p.product_id = x.product_id
    WHERE f.has_review AND f.order_status = 'delivered' AND f.days_late <= 0
    GROUP BY p.product_category
""")
ref = cats["unhappy_on_time"].sum() / cats["reviewed_on_time"].sum()   # usual on-time rate
needed = math.ceil(1.96 ** 2 * ref * (1 - ref) / 0.025 ** 2)            # n for a ±2.5-point interval
MIN_GROUP = 500
print(f"usual on-time unhappy rate {100 * ref:.2f}%; orders needed for ±2.5 points: {needed}; "
      f"rule used: {MIN_GROUP}+ orders per group")

usual on-time unhappy rate 8.48%; orders needed for ±2.5 points: 478; rule used: 500+ orders per group


**Many tests at once.** Comparing every category with the rest means many tests. At the 0.05 line,
about 1 in 20 tests would look "significant" by luck alone. The **Holm correction** makes each test
stricter, depending on how many tests were run, so the list of category differences doesn't fill up
with flukes.

In [12]:
def holm(p_values: list) -> list:
    """Holm-adjusted p-values: sort, multiply the k-th smallest by (m - k), keep them in order."""
    m, order = len(p_values), sorted(range(len(p_values)), key=lambda i: p_values[i])
    adjusted, running = [0.0] * m, 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p_values[i]))
        adjusted[i] = running
    return adjusted

big = cats[cats["reviewed_on_time"] >= MIN_GROUP]
rows = []
for r in big.itertuples():
    rest_u, rest_n = cats["unhappy_on_time"].sum() - r.unhappy_on_time, cats["reviewed_on_time"].sum() - r.reviewed_on_time
    rows.append(compare(r.category, int(r.unhappy_on_time), int(r.reviewed_on_time), int(rest_u), int(rest_n)))
cat_tests = pd.DataFrame(rows).set_index("comparison")
cat_tests["p_holm"] = holm(list(cat_tests["p_value"]))
cat_tests["differs"] = cat_tests["p_holm"] < 0.05
print(f"{len(cat_tests)} categories tested; {cat_tests['differs'].sum()} differ from the rest after Holm")
cat_tests.sort_values("gap_points", ascending=False)[
    ["n_a", "rate_a_pct", "rate_b_pct", "gap_points", "gap_low", "gap_high", "p_value", "p_holm", "differs"]]

26 categories tested; 12 differ from the rest after Holm


,n_a,rate_a_pct,rate_b_pct,gap_points,gap_low,gap_high,p_value,p_holm,differs
comparison,,,,,,,,,
office_furniture,1102,17.15,8.37,8.78,6.55,11.01,4.677e-25,1.216e-23,True
unknown,1223,11.94,8.43,3.5,1.68,5.33,1.589e-05,0.0003019,True
computers_accessories,5879,10.53,8.33,2.2,1.39,3,6.292e-09,1.573e-07,True
bed_bath_table,7766,10.19,8.31,1.87,1.17,2.57,1.869e-08,4.486e-07,True
furniture_decor,5399,10.19,8.37,1.82,0.99,2.65,3.898e-06,8.185e-05,True
telephony,3683,10.21,8.41,1.8,0.81,2.8,0.0001374,0.002474,True
small_appliances,568,9.15,8.48,0.68,-1.7,3.06,0.6161,1,False
watches_gifts,4874,8.99,8.45,0.53,-0.29,1.36,0.2031,1,False
electronics,2273,8.84,8.47,0.37,-0.81,1.55,0.5576,1,False


`rate_b_pct` = all other on-time single-product orders. Only categories marked `differs` have a gap
that survives the stricter check; the others may be luck.